# 33 · Comportamento de quem entrou na campanha de agosto de 2026

A campanha sorteou os assinantes ativos em 10/08/2026 em quatro grupos: controle (sem mensagem),
conteúdo (plano de treino personalizado), desafio (desafio de 6 semanas) e cupom (oferta de desconto).
Envio em 10/08/2026 às 10h, por push e e-mail.

Leitura por intenção de tratar (decisoes.md): todos os sorteados de cada grupo contra o controle, sem
filtrar por abertura ou clique. Janela: do envio ao corte (51 dias).

Três fatores: ativos (quem segue assinante no corte), engajamento (níveis de `src/engajamento.py` na
véspera do envio e no corte) e churn (saídas de 10/08 a 30/09/2026 e renovação desligada).

In [1]:
import sys
import numpy as np
import pandas as pd
from pathlib import Path
sys.path.insert(0, "../src")
from engajamento import niveis_engajamento, NIVEIS

pd.set_option("display.width", 250)
pd.set_option("display.max_columns", 40)
pd.set_option("display.float_format", lambda v: f"{v:,.1f}")
P = Path("../dados/processados"); S = Path("../saidas")
ENVIO = pd.Timestamp("2026-08-10"); CORTE = pd.Timestamp("2026-09-30")
ORDEM = ["controle", "conteudo", "desafio", "cupom"]

camp = pd.read_parquet(P / "campanha_limpa.parquet")
assin = pd.read_parquet(P / "assinaturas_limpa.parquet", columns=["user_id", "inicio", "data_fim", "motivo_fim", "ativa_no_corte", "renovacao_automatica"])
resumo = pd.read_parquet(P / "assinaturas_resumo_eventos.parquet", columns=["user_id", "ultimo_desligamento", "renovacao_ligada_no_corte"])
ativ = pd.read_parquet(P / "atividades_limpa.parquet", columns=["user_id", "data"])
c = camp.merge(assin, on="user_id", how="left", validate="one_to_one").merge(resumo, on="user_id", how="left", validate="one_to_one")
print("sorteados:", len(camp), "-> depois das junções:", len(c), "| sem assinatura:", int(c.inicio.isna().sum()))
print("fora da janela do envio (assinou depois das 10h ou saiu antes):", int(c.fora_da_janela_do_envio.sum()), "(mantidos, como em decisoes.md)")
c["grupo"] = pd.Categorical(c.grupo, ORDEM, ordered=True)

def z(p1, n1, p0, n0):
    # diferença de proporções contra o controle: intervalo de 95% e valor-p (normal)
    from math import sqrt, erf
    d = p1 - p0; se = sqrt(p1 * (1 - p1) / n1 + p0 * (1 - p0) / n0)
    pv = 2 * (1 - 0.5 * (1 + erf(abs(d / se) / sqrt(2)))) if se > 0 else np.nan
    return d * 100, (d - 1.96 * se) * 100, (d + 1.96 * se) * 100, pv

def contra_controle(d, col, nome=None):
    g = d.groupby("grupo", observed=True)[col].agg(n="size", casos="sum")
    g["%"] = g.casos / g.n * 100
    p0, n0 = g.loc["controle", "casos"] / g.loc["controle", "n"], g.loc["controle", "n"]
    r = [z(g.loc[k, "casos"] / g.loc[k, "n"], g.loc[k, "n"], p0, n0) if k != "controle" else (np.nan,) * 4 for k in g.index]
    g[["dif. (p.p.)", "IC95 inf", "IC95 sup", "valor-p"]] = r
    g["valor-p"] = g["valor-p"].round(3)
    if nome: print("==", nome)
    print(g); print()
    return g

sorteados: 14120 -> depois das junções: 14120 | sem assinatura: 0
fora da janela do envio (assinou depois das 10h ou saiu antes): 48 (mantidos, como em decisoes.md)


## 1. Quem entrou: tamanho e perfil dos grupos na véspera do envio

In [2]:
t = c.groupby("grupo", observed=True).agg(sorteados=("user_id", "size"), pct_mensal=("plano_no_envio", lambda s: (s == "mensal").mean() * 100),
                                          meses_de_casa=("inicio", lambda s: ((ENVIO - s).dt.days / 30.44).median()))
print(t); print()
print("mensagem de cada grupo"); print(c.groupby("grupo", observed=True).agg(titulo=("titulo_mensagem", "first"), ofertas=("oferta", lambda s: sorted(s.dropna().unique()))).to_string()); print()
print("oferta do cupom por plano"); k = c[c.grupo == "cupom"]; print(pd.crosstab(k.plano_no_envio, k.oferta)); print()

antes = niveis_engajamento(ativ, c.user_id, ENVIO - pd.Timedelta(days=1), inicio=c.set_index("user_id").inicio)
c = c.merge(antes[["user_id", "nivel", "treinos_30d", "dias_sem_treinar"]].rename(columns={"nivel": "nivel_antes", "treinos_30d": "treinos_30d_antes", "dias_sem_treinar": "dias_sem_treinar_antes"}), on="user_id", validate="one_to_one")
print("depois da junção com o nível na véspera:", len(c))
n = pd.crosstab(c.grupo, c.nivel_antes, margins=True, margins_name="total").reindex(columns=NIVEIS + ["total"])
print("nível de engajamento em 09/08/2026 (contas)"); print(n)
print((pd.crosstab(c.grupo, c.nivel_antes, normalize="index", margins=True, margins_name="total") * 100).reindex(columns=NIVEIS)); print()
c["deslig_antes"] = c.ultimo_desligamento.notna() & (c.ultimo_desligamento < ENVIO + pd.Timedelta(hours=10))
print("% com um desligamento da renovação antes do envio:", (c.groupby("grupo", observed=True).deslig_antes.mean() * 100).round(1).to_dict())

          sorteados  pct_mensal  meses_de_casa
grupo                                         
controle       3579        40.2            9.1
conteudo       3554        41.2            8.8
desafio        3515        41.4            9.0
cupom          3472        40.9            9.2

mensagem de cada grupo
                                                 titulo                        ofertas
grupo                                                                                 
controle                                           None                             []
conteudo  Seu plano de treino personalizado está pronto                             []
desafio     Novo desafio de 6 semanas montado para você                             []
cupom                      Oferta para voltar a treinar  [1_mes_gratis, 30pct_3_meses]

oferta do cupom por plano
oferta          1_mes_gratis  30pct_3_meses
plano_no_envio                             
anual                   2052              0
mensal       

depois da junção com o nível na véspera: 14120
nível de engajamento em 09/08/2026 (contas)
nivel_antes  1. alto  2. médio  3. baixo  4. possível churn  total
grupo                                                             
controle        2484       586       125                384   3579
conteudo        2524       558       112                360   3554
desafio         2436       601       119                359   3515
cupom           2420       559       105                388   3472
total           9864      2304       461               1491  14120
nivel_antes  1. alto  2. médio  3. baixo  4. possível churn
grupo                                                      
controle        69.4      16.4       3.5               10.7
conteudo        71.0      15.7       3.2               10.1
desafio         69.3      17.1       3.4               10.2
cupom           69.7      16.1       3.0               11.2
total           69.9      16.3       3.3               10.6

% com um desligamen

## 2. Reação à mensagem: abertura e clique (só grupos tratados; descritivo)

In [3]:
t = c[c.tratado].groupby("grupo", observed=True).agg(enviados=("user_id", "size"), abriu_push=("abriu_push", "mean"), abriu_email=("abriu_email", "mean"), clicou=("clicou", "mean"))
c["abriu"] = c.abriu_push | c.abriu_email
t["abriu_algum"] = c[c.tratado].groupby("grupo", observed=True).abriu.mean()
t.iloc[:, 1:] *= 100; print(t); print()
tr = c[c.tratado]
print("% que abriu e % que clicou, por nível de engajamento na véspera")
g = tr.groupby("nivel_antes").agg(enviados=("user_id", "size"), abriu=("abriu", "mean"), clicou=("clicou", "mean")); g.iloc[:, 1:] *= 100; print(g); print()
print("% que clicou, por grupo e nível na véspera")
print(tr.pivot_table(index="nivel_antes", columns="grupo", values="clicou", aggfunc="mean", observed=True) * 100); print()
print("% que clicou, por grupo e plano")
print(tr.pivot_table(index="plano_no_envio", columns="grupo", values="clicou", aggfunc="mean", observed=True) * 100)

          enviados  abriu_push  abriu_email  clicou  abriu_algum
grupo                                                           
conteudo      3554        32.0         18.0    14.9         44.3
desafio       3515        30.6         18.4    14.3         42.9
cupom         3472        31.6         19.0    16.4         44.6

% que abriu e % que clicou, por nível de engajamento na véspera
                   enviados  abriu  clicou
nivel_antes                               
1. alto                7380   44.1    15.0
2. médio               1718   44.9    17.2
3. baixo                336   46.7    15.8
4. possível churn      1107   40.4    13.6

% que clicou, por grupo e nível na véspera
grupo              conteudo  desafio  cupom
nivel_antes                                
1. alto                15.1     14.4   15.5
2. médio               15.8     15.1   20.8
3. baixo               18.8     10.9   18.1
4. possível churn      11.4     13.4   15.7

% que clicou, por grupo e plano
grupo      

## 3. Churn: quem saiu entre o envio e o corte

In [4]:
c["saiu"] = c.data_fim.between(ENVIO, CORTE)
print("saídas de 10/08 a 30/09/2026:", int(c.saiu.sum()), "| ativos no corte:", int(c.ativa_no_corte.sum()), "| saíram antes do envio:", int((c.data_fim < ENVIO).sum()))
res = {}
res["todos"] = contra_controle(c, "saiu", "todos os sorteados: % que saiu até o corte")
res["mensal"] = contra_controle(c[c.plano_no_envio == "mensal"], "saiu", "MENSAL: % que saiu até o corte")
res["anual com aniversário"] = contra_controle(c[(c.plano_no_envio == "anual") & c.aniversario_na_janela], "saiu", "ANUAL com aniversário de 10/08 a 30/09: % que saiu")
k = c[(c.plano_no_envio == "anual") & ~c.aniversario_na_janela]
print("anual sem aniversário na janela:", len(k), "| saíram:", int(k.saiu.sum())); print()
contra_controle(c[~c.fora_da_janela_do_envio & (c.plano_no_envio == "mensal")], "saiu", "sensibilidade, MENSAL sem as contas fora da janela do envio")
print("tipo de saída por grupo (contas)"); print(pd.crosstab(c[c.saiu].grupo, c[c.saiu].motivo_fim))
pd.concat(res, names=["recorte", "grupo"]).to_csv(S / "campanha_ago26_churn_por_grupo.csv", encoding="utf-8-sig")

saídas de 10/08 a 30/09/2026: 1258 | ativos no corte: 12862 | saíram antes do envio: 0
== todos os sorteados: % que saiu até o corte
             n  casos   %  dif. (p.p.)  IC95 inf  IC95 sup  valor-p
grupo                                                              
controle  3579    352 9.8          NaN       NaN       NaN      NaN
conteudo  3554    325 9.1         -0.7      -2.1       0.7      0.3
desafio   3515    306 8.7         -1.1      -2.5       0.2      0.1
cupom     3472    275 7.9         -1.9      -3.2      -0.6      0.0

== MENSAL: % que saiu até o corte
             n  casos    %  dif. (p.p.)  IC95 inf  IC95 sup  valor-p
grupo                                                               
controle  1440    210 14.6          NaN       NaN       NaN      NaN
conteudo  1463    198 13.5         -1.0      -3.6       1.5      0.4
desafio   1456    162 11.1         -3.5      -5.9      -1.0      0.0
cupom     1420    127  8.9         -5.6      -8.0      -3.3      0.0

== ANUAL 

In [5]:
# Renovação automática: quem desligou depois do envio e quem segue ativo com ela desligada
c["desligou_depois"] = c.ultimo_desligamento.notna() & (c.ultimo_desligamento >= ENVIO + pd.Timedelta(hours=10))
c["ativo_desligada"] = c.ativa_no_corte & (c.renovacao_automatica == "nao")
c["saiu_ou_desligada"] = c.saiu | c.ativo_desligada
contra_controle(c, "desligou_depois", "todos: % com desligamento da renovação depois do envio")
contra_controle(c[c.plano_no_envio == "anual"], "saiu_ou_desligada", "ANUAL: % que saiu ou está ativo com a renovação desligada no corte")
contra_controle(c[c.plano_no_envio == "mensal"], "saiu_ou_desligada", "MENSAL: % que saiu ou está ativo com a renovação desligada no corte")
m = c[(c.plano_no_envio == "mensal") & c.saiu].assign(semana=lambda d: ((d.data_fim - ENVIO).dt.days // 7 + 1).clip(upper=8))
print("MENSAL: saídas por semana depois do envio"); print(pd.crosstab(m.semana, m.grupo))

== todos: % com desligamento da renovação depois do envio
             n  casos   %  dif. (p.p.)  IC95 inf  IC95 sup  valor-p
grupo                                                              
controle  3579    326 9.1          NaN       NaN       NaN      NaN
conteudo  3554    313 8.8         -0.3      -1.6       1.0      0.7
desafio   3515    236 6.7         -2.4      -3.6      -1.1      0.0
cupom     3472    283 8.2         -1.0      -2.3       0.4      0.2

== ANUAL: % que saiu ou está ativo com a renovação desligada no corte
             n  casos    %  dif. (p.p.)  IC95 inf  IC95 sup  valor-p
grupo                                                               
controle  2139    650 30.4          NaN       NaN       NaN      NaN
conteudo  2091    662 31.7          1.3      -1.5       4.1      0.4
desafio   2059    629 30.5          0.2      -2.6       2.9      0.9
cupom     2052    687 33.5          3.1       0.3       5.9      0.0

== MENSAL: % que saiu ou está ativo com a renova

## 4. Engajamento: antes do envio e no corte

In [6]:
depois = niveis_engajamento(ativ, c.user_id, CORTE, inicio=c.set_index("user_id").inicio)
c = c.merge(depois[["user_id", "nivel", "treinos_30d"]].rename(columns={"nivel": "nivel_corte", "treinos_30d": "treinos_30d_corte"}), on="user_id", validate="one_to_one")
jan = ativ[(ativ.data > ENVIO) & (ativ.data <= CORTE) & ativ.user_id.isin(c.user_id)].groupby("user_id").size().rename("treinos_pos")
c = c.join(jan, on="user_id"); c["treinos_pos"] = c.treinos_pos.fillna(0)
print("depois das junções:", len(c))
c["alto_antes"] = c.nivel_antes == NIVEIS[0]; c["alto_corte"] = c.nivel_corte == NIVEIS[0]
c["parado_antes"] = c.nivel_antes.isin(NIVEIS[2:]); c["parado_corte"] = c.nivel_corte.isin(NIVEIS[2:])
print("todos os sorteados (inclui quem saiu: o treino continua sendo registrado para ex-assinantes)")
t = c.groupby("grupo", observed=True).agg(n=("user_id", "size"), alto_antes=("alto_antes", "mean"), alto_corte=("alto_corte", "mean"), parado_antes=("parado_antes", "mean"), parado_corte=("parado_corte", "mean"),
                                          treinos_30d_antes=("treinos_30d_antes", "mean"), treinos_30d_corte=("treinos_30d_corte", "mean"), treinos_51d_pos=("treinos_pos", "mean"))
t[["alto_antes", "alto_corte", "parado_antes", "parado_corte"]] *= 100; print(t); print()
contra_controle(c, "alto_corte", "% no nível alto no corte, todos os sorteados")
at = c[c.ativa_no_corte]
print("só quem segue ativo no corte:", len(at))
n = pd.crosstab(at.grupo, at.nivel_corte).reindex(columns=NIVEIS)
print((n.div(n.sum(axis=1), axis=0) * 100).join(n.sum(axis=1).rename("ativos")).join(at.groupby("grupo", observed=True).treinos_30d_corte.mean().rename("treinos 30d"))); print()
print("quem estava parado (baixo ou possível churn) na véspera: % que voltou a treinar nos 51 dias")
p0 = c[c.parado_antes].assign(voltou=lambda d: d.treinos_pos > 0)
contra_controle(p0, "voltou")

depois das junções: 14120
todos os sorteados (inclui quem saiu: o treino continua sendo registrado para ex-assinantes)
             n  alto_antes  alto_corte  parado_antes  parado_corte  treinos_30d_antes  treinos_30d_corte  treinos_51d_pos
grupo                                                                                                                    
controle  3579        69.4        63.0          14.2          20.0                8.3                8.4             14.4
conteudo  3554        71.0        64.4          13.3          17.9                8.3                8.5             14.6
desafio   3515        69.3        66.3          13.6          16.6                8.3                8.9             14.9
cupom     3472        69.7        64.7          14.2          18.0                8.3                8.6             14.7

== % no nível alto no corte, todos os sorteados
             n  casos    %  dif. (p.p.)  IC95 inf  IC95 sup  valor-p
grupo                          

,n,casos,%,dif. (p.p.),IC95 inf,IC95 sup,valor-p
grupo,,,,,,,
controle,509,211,41.5,NaN,NaN,NaN,NaN
conteudo,472,202,42.8,1.3,-4.8,7.5,0.7
desafio,478,253,52.9,11.5,5.3,17.7,0.0
cupom,493,217,44.0,2.6,-3.6,8.7,0.4


## 5. O efeito muda conforme o engajamento antes do envio?

In [7]:
het = {}
for plano in ["mensal", "anual"]:
    d = c[c.plano_no_envio == plano]
    desfecho = "saiu" if plano == "mensal" else "saiu_ou_desligada"
    t = d.pivot_table(index="nivel_antes", columns="grupo", values=desfecho, aggfunc="mean", observed=True) * 100
    n = d.pivot_table(index="nivel_antes", columns="grupo", values=desfecho, aggfunc="size", observed=True)
    for g in ORDEM[1:]:
        t[f"{g} − controle"] = t[g] - t["controle"]
    het[plano] = t
    print(f"== {plano.upper()}: % com o desfecho '{desfecho}', por nível na véspera e grupo"); print(t.round(1)); print(n); print()
pd.concat(het, names=["plano", "nivel_antes"]).to_csv(S / "campanha_ago26_churn_por_nivel_antes.csv", encoding="utf-8-sig")
# dentro do mensal médio/baixo: teste por grupo
contra_controle(c[(c.plano_no_envio == "mensal") & (c.nivel_antes != NIVEIS[0])], "saiu", "MENSAL fora do nível alto na véspera: % que saiu")
contra_controle(c[(c.plano_no_envio == "mensal") & (c.nivel_antes == NIVEIS[0])], "saiu", "MENSAL no nível alto na véspera: % que saiu")

== MENSAL: % com o desfecho 'saiu', por nível na véspera e grupo
grupo              controle  conteudo  desafio  cupom  conteudo − controle  desafio − controle  cupom − controle
nivel_antes                                                                                                     
1. alto                11.6      10.4      8.4    6.3                 -1.2                -3.2              -5.3
2. médio               30.7      27.9     22.5   23.4                 -2.8                -8.2              -7.3
3. baixo               27.3      80.0     77.8   71.4                 52.7                50.5              44.2
4. possível churn     100.0     100.0     40.0    0.0                  0.0               -60.0            -100.0
grupo              controle  conteudo  desafio  cupom
nivel_antes                                          
1. alto                1224      1242     1215   1215
2. médio                202       208      227    197
3. baixo                 11        10    

,n,casos,%,dif. (p.p.),IC95 inf,IC95 sup,valor-p
grupo,,,,,,,
controle,1224,142,11.6,NaN,NaN,NaN,NaN
conteudo,1242,129,10.4,-1.2,-3.7,1.3,0.3
desafio,1215,102,8.4,-3.2,-5.6,-0.8,0.0
cupom,1215,76,6.3,-5.3,-7.6,-3.1,0.0


## 6. Quem abriu ou clicou (descritivo: não mede efeito)

In [8]:
tr = c[c.tratado].assign(reacao=lambda d: np.where(d.clicou, "3. clicou", np.where(d.abriu, "2. abriu, não clicou", "1. não abriu")))
g = tr.groupby(["plano_no_envio", "reacao"]).agg(enviados=("user_id", "size"), saiu=("saiu", "mean"), alto_antes=("alto_antes", "mean"), alto_corte=("alto_corte", "mean"), treinos_pos=("treinos_pos", "mean"))
g[["saiu", "alto_antes", "alto_corte"]] *= 100; print(g)
print()
print("controle, para referência"); k = c[c.grupo == "controle"].groupby("plano_no_envio").agg(n=("user_id", "size"), saiu=("saiu", "mean"), alto_antes=("alto_antes", "mean"), alto_corte=("alto_corte", "mean"), treinos_pos=("treinos_pos", "mean"))
k[["saiu", "alto_antes", "alto_corte"]] *= 100; print(k)

                                     enviados  saiu  alto_antes  alto_corte  treinos_pos
plano_no_envio reacao                                                                   
anual          1. não abriu              3483   7.0        59.7        55.8         12.4
               2. abriu, não clicou      1779   6.7        60.5        57.4         12.7
               3. clicou                  940   6.1        58.8        59.1         13.3
mensal         1. não abriu              2426  11.0        84.3        77.7         17.7
               2. abriu, não clicou      1249  11.4        85.9        77.3         18.1
               3. clicou                  664  11.7        83.3        74.7         17.3

controle, para referência
                   n  saiu  alto_antes  alto_corte  treinos_pos
plano_no_envio                                                 
anual           2139   6.6        58.9        54.8         12.0
mensal          1440  14.6        85.0        75.1         17.8


## 7. O grupo cupom depois do desconto: o que os dados deixam ver

O desconto do mensal vale por 3 meses a partir de 10/08/2026, ou seja, até novembro. O corte é 30/09/2026.

In [9]:
pag = pd.read_parquet(P / "pagamentos_limpo.parquet", columns=["user_id", "data_br", "status", "valor", "cupom"])
print("última cobrança no arquivo:", pag.data_br.max())
pos = pag[(pag.data_br >= ENVIO + pd.Timedelta(hours=10)) & (pag.status == "aprovado")].merge(c[["user_id", "grupo", "plano_no_envio"]], on="user_id")
print("cobranças aprovadas dos sorteados depois do envio:", len(pos))
print("valor cobrado por grupo e plano (nº de cobranças)")
print(pd.crosstab([pos.plano_no_envio, pos.grupo], pos.valor))
print()
n = pos.groupby("user_id").size().rename("cobrancas_pos")
k = c.join(n, on="user_id").fillna({"cobrancas_pos": 0})
m = k[k.plano_no_envio == "mensal"]
print("MENSAL: nº de cobranças aprovadas depois do envio, por grupo (% da linha)")
print((pd.crosstab(m.grupo, m.cobrancas_pos.astype(int), normalize="index") * 100).round(1))
print()
at = m[m.ativa_no_corte]
g = at.groupby("grupo", observed=True).agg(ativos=("user_id", "size"), desligada=("ativo_desligada", "sum"))
g["% com renovação desligada"] = g.desligada / g.ativos * 100
print("MENSAL ativo no corte: renovação automática desligada (saída anunciada)"); print(g)
print()
a = k[(k.plano_no_envio == "anual") & k.ativa_no_corte]
g = a.groupby("grupo", observed=True).agg(ativos=("user_id", "size"), desligada=("ativo_desligada", "sum"))
g["% com renovação desligada"] = g.desligada / g.ativos * 100
print("ANUAL ativo no corte: renovação automática desligada"); print(g)
print()
print("dias pagos cobertos depois do corte: fim do desconto do mensal =", (ENVIO + pd.DateOffset(months=3)).date(), "| dias do corte até lá:", ((ENVIO + pd.DateOffset(months=3)) - CORTE).days)

última cobrança no arquivo: 2026-09-30 22:54:51
cobranças aprovadas dos sorteados depois do envio: 9615
valor cobrado por grupo e plano (nº de cobranças)
valor                    39.9   239.9  299.9
plano_no_envio grupo                        
anual          controle      0     10    171
               conteudo      0      9    153
               desafio       0      9    162
               cupom         0      8    138
mensal         controle   2196      0      0
               conteudo   2222      0      0
               desafio    2274      0      0
               cupom      2263      0      0

MENSAL: nº de cobranças aprovadas depois do envio, por grupo (% da linha)
cobrancas_pos   0    1    2   3
grupo                          
controle      8.9 29.7 61.4 0.0
conteudo      9.6 28.9 61.4 0.1
desafio       7.8 28.2 64.0 0.0
cupom         6.8 27.0 66.1 0.1

MENSAL ativo no corte: renovação automática desligada (saída anunciada)
          ativos  desligada  % com renovação desligada
g

ANUAL ativo no corte: renovação automática desligada
          ativos  desligada  % com renovação desligada
grupo                                                 
controle    1997        508                       25.4
conteudo    1964        535                       27.2
desafio     1915        485                       25.3
cupom       1904        539                       28.3

dias pagos cobertos depois do corte: fim do desconto do mensal = 2026-11-10 | dias do corte até lá: 41
